<a href="https://colab.research.google.com/github/ankitaguptacs24-ai/AI_LAB/blob/main/LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8-Puzzle Solver using A* Search
This notebook implements the A* search algorithm to solve the classic 8-puzzle. We will compare two different heuristic functions:
1. **Misplaced Tiles** (Hamming Distance)
2. **Manhattan Distance**

In [1]:
import heapq
import copy

# Define the goal state for the 8-puzzle
GOAL_STATE = (1, 2, 3, 8, 0, 4, 7, 6, 5)  # 0 represents the empty space

class PuzzleState:
    def __init__(self, board, g=0, parent=None):
        self.board = tuple(board)  # Use tuple for immutability and hashing
        self.g = g                 # Path cost from start to current node
        self.parent = parent
        self.f = 0                 # Total estimated cost (g + h)

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_index(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        idx = self.get_blank_index()
        row, col = idx // 3, idx % 3

        # Define possible moves: Up, Down, Left, Right
        moves = []
        if row > 0: moves.append(-3) # Up
        if row < 2: moves.append(3)  # Down
        if col > 0: moves.append(-1) # Left
        if col < 2: moves.append(1)  # Right

        for move in moves:
            new_board = list(self.board)
            new_idx = idx + move
            # Swap
            new_board[idx], new_board[new_idx] = new_board[new_idx], new_board[idx]
            neighbors.append(PuzzleState(new_board, self.g + 1, self))

        return neighbors

In [2]:
def heuristic_misplaced_tiles(board):
    """Calculates the number of misplaced tiles compared to the goal state."""
    misplaced = 0
    for i in range(len(board)):
        if board[i] != 0 and board[i] != GOAL_STATE[i]:
            misplaced += 1
    return misplaced

def heuristic_manhattan_distance(board):
    """Calculates the sum of Manhattan distances of the tiles from their goal positions."""
    distance = 0
    for i in range(len(board)):
        value = board[i]
        if value != 0:
            # Current position
            curr_row, curr_col = i // 3, i % 3
            # Target position in goal state
            target_idx = GOAL_STATE.index(value)
            target_row, target_col = target_idx // 3, target_idx % 3

            distance += abs(curr_row - target_row) + abs(curr_col - target_col)
    return distance

In [3]:
def a_star_search(start_board, heuristic_func):
    """Solves the 8-puzzle using A* search with the given heuristic function."""
    start_state = PuzzleState(start_board)
    start_state.f = start_state.g + heuristic_func(start_state.board)

    open_set = []
    heapq.heappush(open_set, start_state)
    closed_set = set()

    nodes_expanded = 0

    while open_set:
        current_state = heapq.heappop(open_set)
        nodes_expanded += 1

        if current_state.board == GOAL_STATE:
            # Reconstruct path
            path = []
            curr = current_state
            while curr:
                path.append(curr.board)
                curr = curr.parent
            return path[::-1], nodes_expanded

        closed_set.add(current_state.board)

        for neighbor in current_state.get_neighbors():
            if neighbor.board in closed_set:
                continue

            neighbor.f = neighbor.g + heuristic_func(neighbor.board)
            heapq.heappush(open_set, neighbor)

    return None, nodes_expanded

In [4]:
def print_board(board):
    for i in range(0, 9, 3):
        print(f"{board[i]} {board[i+1]} {board[i+2]}")
    print()

# Define an initial solvable state
# (1, 2, 3, 0, 4, 6, 7, 5, 8) can be solved in a few moves
initial_board = (2, 8, 3, 1, 6, 4, 7, 0, 5)

print("--- Solving with Misplaced Tiles Heuristic ---")
path_misplaced, nodes_misplaced = a_star_search(initial_board, heuristic_misplaced_tiles)
print(f"Solvable in: {len(path_misplaced) - 1} moves")
print(f"Nodes expanded: {nodes_misplaced}")

print("\n--- Solving with Manhattan Distance Heuristic ---")
path_manhattan, nodes_manhattan = a_star_search(initial_board, heuristic_manhattan_distance)
print(f"Solvable in: {len(path_manhattan) - 1} moves")
print(f"Nodes expanded: {nodes_manhattan}")

print("\n--- Show Solution Steps (Manhattan) ---")
for step, state in enumerate(path_manhattan):
    print(f"Step {step}:")
    print_board(state)

--- Solving with Misplaced Tiles Heuristic ---
Solvable in: 5 moves
Nodes expanded: 7

--- Solving with Manhattan Distance Heuristic ---
Solvable in: 5 moves
Nodes expanded: 6

--- Show Solution Steps (Manhattan) ---
Step 0:
2 8 3
1 6 4
7 0 5

Step 1:
2 8 3
1 0 4
7 6 5

Step 2:
2 0 3
1 8 4
7 6 5

Step 3:
0 2 3
1 8 4
7 6 5

Step 4:
1 2 3
0 8 4
7 6 5

Step 5:
1 2 3
8 0 4
7 6 5

